In [1]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

# Load the CSV file
csv_file = Path('history_EURUSDm_5m_4d.csv')
df = pd.read_csv(csv_file)

# Display the structure of the CSV to understand the format
print("CSV file info:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)

CSV file info:
Shape: (1148, 6)
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

First few rows:
                        Date     Open     High      Low    Close  Volume
0  2025-11-17 05:55:00+00:00  1.15998  1.16003  1.15988  1.16001      88
1  2025-11-17 06:00:00+00:00  1.16003  1.16003  1.15989  1.15992      76
2  2025-11-17 06:05:00+00:00  1.15991  1.15991  1.15948  1.15953     157
3  2025-11-17 06:10:00+00:00  1.15951  1.15976  1.15948  1.15971      88
4  2025-11-17 06:15:00+00:00  1.15969  1.15984  1.15962  1.15983      84

Data types:
Date       object
Open      float64
High      float64
Low       float64
Close     float64
Volume      int64
dtype: object


In [2]:
# Function to convert CSV to the desired JSON format
def convert_csv_to_json(df, date_column=None):
    """
    Convert CSV DataFrame to JSON format compatible with your trading data
    
    Args:
        df: DataFrame with OHLCV data
        date_column: Name of the date/time column (auto-detected if None)
    
    Returns:
        List of dictionaries in the required format
    """
    # Make a copy to avoid modifying the original
    df_clean = df.copy()
    
    # Auto-detect date column if not specified
    if date_column is None:
        # Common date column names
        date_candidates = ['date', 'time', 'datetime', 'timestamp', 'Date', 'Time', 'DateTime', 'Timestamp']
        for col in date_candidates:
            if col in df_clean.columns:
                date_column = col
                break
        
        # If still not found, try first column
        if date_column is None:
            date_column = df_clean.columns[0]
            print(f"Using first column '{date_column}' as date column")
    
    print(f"Using '{date_column}' as date column")
    
    # Convert date column to datetime
    df_clean[date_column] = pd.to_datetime(df_clean[date_column], utc=True, errors='coerce')
    
    # Standardize column names (case-insensitive mapping)
    column_mapping = {}
    for col in df_clean.columns:
        col_lower = col.lower()
        if col_lower in ['open', 'o']:
            column_mapping[col] = 'Open'
        elif col_lower in ['high', 'h']:
            column_mapping[col] = 'High'
        elif col_lower in ['low', 'l']:
            column_mapping[col] = 'Low'
        elif col_lower in ['close', 'c']:
            column_mapping[col] = 'Close'
        elif col_lower in ['volume', 'vol', 'v']:
            column_mapping[col] = 'Volume'
        elif col == date_column:
            column_mapping[col] = 'Date'
    
    # Rename columns
    df_clean = df_clean.rename(columns=column_mapping)
    
    # Ensure we have all required columns
    required_cols = ['Date', 'Open', 'High', 'Low', 'Close']
    missing_cols = [col for col in required_cols if col not in df_clean.columns]
    if missing_cols:
        print(f"Warning: Missing columns: {missing_cols}")
    
    # Add Volume column if missing
    if 'Volume' not in df_clean.columns:
        df_clean['Volume'] = 0.0
    
    # Select and reorder columns
    final_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
    available_cols = [col for col in final_cols if col in df_clean.columns]
    df_final = df_clean[available_cols].copy()
    
    # Convert to the desired format
    records = []
    for _, row in df_final.iterrows():
        record = {
            'Date': row['Date'].strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(row['Date']) else None,
            'Open': float(row['Open']) if pd.notna(row['Open']) else 0.0,
            'High': float(row['High']) if pd.notna(row['High']) else 0.0,
            'Low': float(row['Low']) if pd.notna(row['Low']) else 0.0,
            'Close': float(row['Close']) if pd.notna(row['Close']) else 0.0,
            'Volume': float(row['Volume']) if pd.notna(row['Volume']) else 0.0
        }
        # Only add records with valid dates
        if record['Date'] is not None:
            records.append(record)
    
    print(f"Converted {len(records)} records")
    return records

# Test the conversion function
json_data = convert_csv_to_json(df)

Using 'Date' as date column
Converted 1148 records


In [4]:
# Display sample of converted data
print("Sample of converted JSON data:")
print(json.dumps(json_data[:3], indent=2))
print(f"\n... and {len(json_data)-3} more records")

# Save to JSON file
output_file = Path('data/history_EURUSDm_5m_4d.json')
output_file.parent.mkdir(exist_ok=True)

with open(output_file, 'w') as f:
    json.dump(json_data, f, indent=2)

print(f"\nData saved to: {output_file}")
print(f"File size: {output_file.stat().st_size / 1024:.1f} KB")

Sample of converted JSON data:
[
  {
    "Date": "2025-11-17 05:55:00+0000",
    "Open": 1.15998,
    "High": 1.16003,
    "Low": 1.15988,
    "Close": 1.16001,
    "Volume": 88.0
  },
  {
    "Date": "2025-11-17 06:00:00+0000",
    "Open": 1.16003,
    "High": 1.16003,
    "Low": 1.15989,
    "Close": 1.15992,
    "Volume": 76.0
  },
  {
    "Date": "2025-11-17 06:05:00+0000",
    "Open": 1.15991,
    "High": 1.15991,
    "Low": 1.15948,
    "Close": 1.15953,
    "Volume": 157.0
  }
]

... and 1145 more records

Data saved to: data/history_EURUSDm_5m_4d.json
File size: 170.7 KB


In [9]:
# Load and examine the account orders history CSV
orders_csv_file = Path('account_orders_history.csv')
orders_df = pd.read_csv(orders_csv_file)

print("Account Orders CSV info:")
print(f"Shape: {orders_df.shape}")
print(f"Columns: {list(orders_df.columns)}")
print("\nFirst few rows:")
print(orders_df.head())
print("\nData types:")
print(orders_df.dtypes)

Account Orders CSV info:
Shape: (34, 24)
Columns: ['ticket', 'time_setup', 'time_setup_msc', 'time_done', 'time_done_msc', 'time_expiration', 'type', 'type_time', 'type_filling', 'state', 'magic', 'position_id', 'position_by_id', 'reason', 'volume_initial', 'volume_current', 'price_open', 'sl', 'tp', 'price_current', 'price_stoplimit', 'symbol', 'comment', 'external_id']

First few rows:
       ticket                 time_setup  time_setup_msc   time_done  \
0  1946904930  2025-11-17 06:09:54+00:00   1763359794666  1763359794   
1  1946909363  2025-11-17 06:10:59+00:00   1763359859779  1763359859   
2  1947497356  2025-11-17 08:13:57+00:00   1763367237972  1763367238   
3  1947497445  2025-11-17 08:13:59+00:00   1763367239647  1763367239   
4  1948171719  2025-11-17 10:03:00+00:00   1763373780996  1763373781   

   time_done_msc  time_expiration  type  type_time  type_filling  state  ...  \
0  1763359794753                0     0          0             1      4  ...   
1  1763359859859

In [11]:
def convert_orders_to_trades_detail(df):
    """
    Convert account orders CSV to trades_detail JSON format
    
    Args:
        df: DataFrame with account orders data
    
    Returns:
        List of dictionaries in trades_detail format
    """
    df_clean = df.copy()
    
    # Common column mappings (case-insensitive)
    column_mappings = {
        # Direction/Type mappings
        'type': 'direction', 'side': 'direction', 'action': 'direction', 'order_type': 'direction',
        'buy_sell': 'direction', 'transaction_type': 'direction',
        
        # Entry mappings
        'open_time': 'entry_time', 'entry_time': 'entry_time', 'start_time': 'entry_time',
        'order_time': 'entry_time', 'execution_time': 'entry_time', 'fill_time': 'entry_time',
        'time': 'entry_time', 'datetime': 'entry_time', 'timestamp': 'entry_time',
        'open_price': 'entry_price', 'entry_price': 'entry_price', 'price': 'entry_price',
        'fill_price': 'entry_price', 'execution_price': 'entry_price',
        
        # Exit mappings  
        'close_time': 'exit_time', 'exit_time': 'exit_time', 'end_time': 'exit_time',
        'close_price': 'exit_price', 'exit_price': 'exit_price',
        
        # Stop/TP mappings
        'stop_loss': 'stop_price', 'sl': 'stop_price', 'stop_price': 'stop_price',
        'take_profit': 'take_profit_price', 'tp': 'take_profit_price', 'target_price': 'take_profit_price',
        
        # Size mappings
        'volume': 'size', 'lot_size': 'size', 'quantity': 'size', 'amount': 'size',
        'position_size': 'size', 'units': 'size',
        
        # P&L mappings
        'profit': 'pnl', 'pnl': 'pnl', 'profit_loss': 'pnl', 'net_profit': 'pnl',
        'realized_pnl': 'pnl', 'gain_loss': 'pnl'
    }
    
    # Create reverse mapping for case-insensitive lookup
    reverse_mapping = {}
    for col in df_clean.columns:
        col_lower = col.lower().replace(' ', '_').replace('-', '_')
        if col_lower in column_mappings:
            reverse_mapping[col] = column_mappings[col_lower]
    
    # Apply mappings
    df_mapped = df_clean.rename(columns=reverse_mapping)
    
    print("Column mappings applied:")
    for old_col, new_col in reverse_mapping.items():
        print(f"  {old_col} -> {new_col}")
    
    # Debug: Show all columns after mapping
    print(f"\nColumns after mapping: {list(df_mapped.columns)}")
    print(f"Sample row data:")
    if not df_mapped.empty:
        sample_row = df_mapped.iloc[0]
        for col in df_mapped.columns:
            print(f"  {col}: {sample_row[col]} (type: {type(sample_row[col])})")
    
    # Convert datetime columns
    datetime_cols = ['entry_time', 'exit_time']
    for col in datetime_cols:
        if col in df_mapped.columns:
            print(f"\nConverting {col} to datetime...")
            original_values = df_mapped[col].head(3).tolist()
            df_mapped[col] = pd.to_datetime(df_mapped[col], utc=True, errors='coerce')
            converted_values = df_mapped[col].head(3).tolist()
            print(f"  Original: {original_values}")
            print(f"  Converted: {converted_values}")
    
    # Standardize direction values
    if 'direction' in df_mapped.columns:
        print(f"\nOriginal direction values: {df_mapped['direction'].unique()}")
        direction_map = {
            'buy': 'BUY', 'b': 'BUY', '1': 'BUY', 'long': 'BUY',
            'sell': 'SELL', 's': 'SELL', '0': 'SELL', 'short': 'SELL',
            'buy_market': 'BUY', 'sell_market': 'SELL'
        }
        df_mapped['direction'] = df_mapped['direction'].astype(str).str.lower().map(direction_map).fillna(df_mapped['direction'])
        print(f"Standardized direction values: {df_mapped['direction'].unique()}")
    
    # Initialize equity tracking
    starting_equity = 100.0
    current_equity = starting_equity
    
    # Convert to trades_detail format
    trades_detail = []
    
    print(f"\nProcessing {len(df_mapped)} rows...")
    
    for idx, row in df_mapped.iterrows():
        trade = {}
        
        # Required fields with fallbacks
        trade['direction'] = row.get('direction', 'BUY') if pd.notna(row.get('direction')) else 'BUY'
        
        # Entry time - be more flexible
        entry_time_value = row.get('entry_time')
        if pd.notna(entry_time_value):
            if isinstance(entry_time_value, str):
                trade['entry_time'] = entry_time_value  # Already a string
            else:
                trade['entry_time'] = entry_time_value.strftime('%Y-%m-%dT%H:%M:%S%z')
        else:
            # Try to use any available time column
            time_cols = [col for col in df_mapped.columns if 'time' in col.lower() or 'date' in col.lower()]
            if time_cols:
                time_val = row.get(time_cols[0])
                if pd.notna(time_val):
                    try:
                        parsed_time = pd.to_datetime(time_val, utc=True)
                        trade['entry_time'] = parsed_time.strftime('%Y-%m-%dT%H:%M:%S%z')
                    except:
                        trade['entry_time'] = str(time_val)
            
        # Entry price - be more flexible
        entry_price_value = row.get('entry_price')
        if pd.notna(entry_price_value):
            trade['entry_price'] = float(entry_price_value)
        else:
            # Try to find any price column
            price_cols = [col for col in df_mapped.columns if 'price' in col.lower()]
            if price_cols:
                price_val = row.get(price_cols[0])
                if pd.notna(price_val):
                    trade['entry_price'] = float(price_val)
        
        # Stop price
        if pd.notna(row.get('stop_price')):
            trade['stop_price'] = float(row['stop_price'])
        
        # Take profit price
        if pd.notna(row.get('take_profit_price')):
            trade['take_profit_price'] = float(row['take_profit_price'])
        
        # Exit time
        if pd.notna(row.get('exit_time')):
            if isinstance(row['exit_time'], str):
                trade['exit_time'] = row['exit_time']
            else:
                trade['exit_time'] = row['exit_time'].strftime('%Y-%m-%dT%H:%M:%S%z')
        
        # Exit price
        if pd.notna(row.get('exit_price')):
            trade['exit_price'] = float(row['exit_price'])
        
        # Size
        if pd.notna(row.get('size')):
            trade['size'] = float(row['size'])
        else:
            trade['size'] = 0.01  # Default small size
        
        # Calculate stop distance in pips (if we have entry and stop prices)
        if 'stop_price' in trade and trade.get('entry_price', 0) > 0:
            pip_value = 0.0001  # Standard for most forex pairs
            if trade['direction'] == 'BUY':
                stop_distance = (trade['entry_price'] - trade['stop_price']) / pip_value
            else:
                stop_distance = (trade['stop_price'] - trade['entry_price']) / pip_value
            trade['stop_distance_pips'] = round(stop_distance, 2)
        
        # P&L
        if pd.notna(row.get('pnl')):
            trade['pnl'] = float(row['pnl'])
            current_equity += trade['pnl']
            trade['equity_after'] = round(current_equity, 2)
        
        # Size calculations (if not available, use defaults)
        if 'size' in trade:
            trade['size_pre_cap'] = trade['size']
            trade['size_after_leverage_cap'] = trade['size'] 
            trade['size_after_rounding'] = trade['size']
            trade['size_after_max_cap'] = trade['size']
        
        # More lenient validation - include if we have either time or price
        has_entry_time = 'entry_time' in trade and trade['entry_time'] is not None
        has_entry_price = 'entry_price' in trade and trade.get('entry_price', 0) > 0
        
        if has_entry_time or has_entry_price:
            trades_detail.append(trade)
            if len(trades_detail) <= 3:  # Show first few trades for debugging
                print(f"Trade {len(trades_detail)}: {trade}")
    
    print(f"\nConverted {len(trades_detail)} trades")
    return trades_detail

# Convert the orders data
trades_detail = convert_orders_to_trades_detail(orders_df)

Column mappings applied:
  type -> direction
  sl -> stop_price
  tp -> take_profit_price

Columns after mapping: ['ticket', 'time_setup', 'time_setup_msc', 'time_done', 'time_done_msc', 'time_expiration', 'direction', 'type_time', 'type_filling', 'state', 'magic', 'position_id', 'position_by_id', 'reason', 'volume_initial', 'volume_current', 'price_open', 'stop_price', 'take_profit_price', 'price_current', 'price_stoplimit', 'symbol', 'comment', 'external_id']
Sample row data:
  ticket: 1946904930 (type: <class 'numpy.int64'>)
  time_setup: 2025-11-17 06:09:54+00:00 (type: <class 'str'>)
  time_setup_msc: 1763359794666 (type: <class 'numpy.int64'>)
  time_done: 1763359794 (type: <class 'numpy.int64'>)
  time_done_msc: 1763359794753 (type: <class 'numpy.int64'>)
  time_expiration: 0 (type: <class 'numpy.int64'>)
  direction: 0 (type: <class 'numpy.int64'>)
  type_time: 0 (type: <class 'numpy.int64'>)
  type_filling: 1 (type: <class 'numpy.int64'>)
  state: 4 (type: <class 'numpy.int64'

In [12]:
# Display sample of converted trades_detail
print("Sample of converted trades_detail:")
if trades_detail:
    print(json.dumps(trades_detail[:2], indent=2))
    print(f"\n... and {len(trades_detail)-2} more trades")
else:
    print("No trades converted - check column mappings")

# Create the full JSON structure
result_json = {
    "trades_detail": trades_detail
}

# Save to JSON file
trades_output_file = Path('data/account_orders_trades_detail.json')
trades_output_file.parent.mkdir(exist_ok=True)

with open(trades_output_file, 'w') as f:
    json.dump(result_json, f, indent=2)

print(f"\nTrades data saved to: {trades_output_file}")
print(f"File size: {trades_output_file.stat().st_size / 1024:.1f} KB")

Sample of converted trades_detail:
[
  {
    "direction": "SELL",
    "entry_time": "2025-11-17T06:09:54+0000",
    "entry_price": 0.0,
    "stop_price": 0.0,
    "take_profit_price": 0.0,
    "size": 0.01,
    "size_pre_cap": 0.01,
    "size_after_leverage_cap": 0.01,
    "size_after_rounding": 0.01,
    "size_after_max_cap": 0.01
  },
  {
    "direction": "BUY",
    "entry_time": "2025-11-17T06:10:59+0000",
    "entry_price": 0.0,
    "stop_price": 0.0,
    "take_profit_price": 0.0,
    "size": 0.01,
    "size_pre_cap": 0.01,
    "size_after_leverage_cap": 0.01,
    "size_after_rounding": 0.01,
    "size_after_max_cap": 0.01
  }
]

... and 32 more trades

Trades data saved to: data/account_orders_trades_detail.json
File size: 11.1 KB


In [13]:
# Validate the converted trades data
def validate_trades_detail(json_file):
    """Load and validate the trades_detail JSON format"""
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    trades_detail = data.get('trades_detail', [])
    
    if not trades_detail:
        print("No trades found in the data")
        return None
    
    trades_df = pd.DataFrame(trades_detail)
    
    print("Trades Detail Validation:")
    print(f"Total trades: {len(trades_df)}")
    
    # Check for datetime columns
    datetime_cols = ['entry_time', 'exit_time']
    for col in datetime_cols:
        if col in trades_df.columns:
            trades_df[col] = pd.to_datetime(trades_df[col], utc=True, errors='coerce')
            valid_dates = trades_df[col].notna().sum()
            print(f"{col}: {valid_dates}/{len(trades_df)} valid dates")
    
    # Check direction distribution
    if 'direction' in trades_df.columns:
        print("Direction distribution:")
        print(trades_df['direction'].value_counts())
    
    # Check for required fields
    required_fields = ['direction', 'entry_time', 'entry_price']
    available_fields = [col for col in required_fields if col in trades_df.columns]
    missing_fields = [col for col in required_fields if col not in trades_df.columns]
    
    print(f"\nAvailable fields: {available_fields}")
    if missing_fields:
        print(f"Missing fields: {missing_fields}")
    
    # Show sample data
    print("\nSample trades:")
    print(trades_df.head(3).to_string())
    
    return trades_df

# Validate the converted trades
validated_trades_df = validate_trades_detail(trades_output_file)

if validated_trades_df is not None:
    print(f"\n✓ Successfully converted {orders_csv_file} to trades_detail format")
    print("The data is now compatible with your trading analysis tools!")
else:
    print("\n⚠ Conversion completed but no valid trades found - check your CSV format")

Trades Detail Validation:
Total trades: 34
entry_time: 34/34 valid dates
Direction distribution:
direction
BUY     19
SELL    15
Name: count, dtype: int64

Available fields: ['direction', 'entry_time', 'entry_price']

Sample trades:
  direction                entry_time  entry_price  stop_price  take_profit_price  size  size_pre_cap  size_after_leverage_cap  size_after_rounding  size_after_max_cap  stop_distance_pips
0      SELL 2025-11-17 06:09:54+00:00          0.0         0.0                0.0  0.01          0.01                     0.01                 0.01                0.01                 NaN
1       BUY 2025-11-17 06:10:59+00:00          0.0         0.0                0.0  0.01          0.01                     0.01                 0.01                0.01                 NaN
2      SELL 2025-11-17 08:13:57+00:00          0.0         0.0                0.0  0.01          0.01                     0.01                 0.01                0.01                 NaN

✓ Successfully